# Notebook 02: Benchmark Prestazionale, Analisi Metriche & Matrici di Confusione
## Fase 2 del Piano Operativo (`results/final_experimental_report.md` - Post-Batch Job)

### Workflow Operativo:
1. **Esecuzione Training e Test via Batch Job**: L'addestramento delle 3 architetture a 50 epoche con Focal Loss ($\gamma=2.0, \alpha=0.5$) a `seed=0` viene eseguito tramite **Batch Job su Jupyter / Cluster DIETI** lanciando:
   ```bash
   bash scripts/launch_batch_training.sh
   # oppure: bash scripts/run_training.sh --all --gpu --epochs 50
   ```
2. **Ruolo di questo Notebook**: Questo notebook si esegue **dopo il completamento del Batch Job** per:
   - Verificare lo stato dei log e l'integrità dei checkpoint (`best_model.pth` in `results/models/`).
   - Caricare i 3 modelli congelati ed effettuare la valutazione comparativa formale sul Test Set (181.366 flussi).
   - Calcolare **Macro F1-Score**, **Balanced Accuracy** e F1 per ciascuna delle 13 classi.
   - Generare e salvare le **3 Matrici di Confusione normalizzate in percentuale (0–100%)** con colorbar condivisa per la pubblicazione.
   - Integrare i dati di efficienza computazionale ed edge latency CPU (`results/metrics/edge_latency_profiling.csv`).
   - Esportare la tabella consuntiva `results/metrics/benchmark_metrics.csv` pronta per `results/final_experimental_report.md`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

for d in [MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from xai.model_loader import load_checkpoint, list_experiments

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso per la valutazione: {device}")

### 2.1 Verifica dello Stato del Batch Job e Checkpoint (Task T2.3)
Ispezioniamo la cartella `results/` per verificare che il batch job abbia completato l'addestramento dei 3 modelli.

In [ ]:
TARGET_MODELS = ['Lopez17CNN', 'BiflowTransformer', 'MedDefenderTransformer']
RUN_IDS = {
    'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
    'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
    'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'
}

print("======================================================================")
print(" VERIFICA RISULTATI DEL BATCH JOB (scripts/run_training.sh)")
print("======================================================================")
all_ready = True
for name, exp_id in RUN_IDS.items():
    best_pth = MODELS_DIR / name / "best_model.pth"
    log_file = RESULTS_DIR / f"train_{exp_id}.log"
    
    status_pth = "✓ Presente" if best_pth.exists() else "✗ Non ancora presente"
    status_log = "✓ Presente" if log_file.exists() else "✗ Non ancora presente"
    
    if not best_pth.exists():
        all_ready = False
        
    print(f"[{name:24s}] Checkpoint: {status_pth} | Log: {status_log}")

if not all_ready:
    print("\n[INFO] Uno o piu modelli non sono ancora stati completati dal Batch Job.")
    print("Per avviare il Batch Job su Jupyter/Cluster, esegui dal terminale:")
    print("    bash scripts/launch_batch_training.sh")
else:
    print("\n[OK] Tutti e 3 i modelli sono stati addestrati con successo dal Batch Job!")

### 2.2 Caricamento dei DataLoader di Test (Task T1.4 & T2.4)
Carichiamo il Test Set fisso a `seed=0` (181.366 flussi) per la valutazione obiettiva *ceteris paribus*.

In [ ]:
from datasets.data_loader import get_loaders

DATA_CANDIDATES = [
    Path('/nas/data/datasets/cic-iomt'),
    REPO_ROOT / 'dataset',
    REPO_ROOT / 'data'
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), REPO_ROOT / 'dataset')

NUM_PKTS = 10
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
NUM_CLASSES = 13

# Read target classes from classes.txt
CLASSES_FILE = DATA_DIR / 'classes.txt'
if CLASSES_FILE.exists():
    with open(CLASSES_FILE) as f:
        CLASS_NAMES = [l.strip() for l in f if l.strip()]
else:
    CLASS_NAMES = [f"Class_{i}" for i in range(NUM_CLASSES)]

_, _, tst_loader, _ = get_loaders(
    datasets=['cic-iomt'],
    num_tasks=1,
    nc_first_task=NUM_CLASSES,
    nc_incr_tasks=0,
    batch_size=128,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    seed=0,
    num_pkts=NUM_PKTS,
    fields=FIELDS
)
test_loader = tst_loader[0]
print(f"[OK] Test DataLoader caricato: {len(test_loader.dataset):,} flussi totali.")
assert len(test_loader.dataset) == 181_366, f"Test set atteso 181.366 flussi (Guida 10.5), trovati {len(test_loader.dataset):,}"

### 2.3 Valutazione Formale dei 3 Modelli sul Test Set (Task T2.4)
Se i checkpoint sono presenti, calcoliamo le predizioni sul test set ed estraiamo le metriche macro e per classe.

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            outputs = model(x)
            if isinstance(outputs, (list, tuple)):
                outputs = outputs[0]
            preds = outputs.argmax(dim=-1).cpu()
            y_true.extend(y.numpy())
            y_pred.extend(preds.numpy())
    return np.array(y_true), np.array(y_pred)

if not all_ready:
    raise RuntimeError("Checkpoint mancanti in results/models/: completare prima il Batch Job di training.")

results = {}
for name, exp_id in RUN_IDS.items():
    print(f"Valutazione in corso: {name}...")
    model, _ = load_checkpoint(exp_name=f"cic-iomt_scratch_{exp_id}", results_dir=str(RESULTS_DIR),
                               network=name, device=str(device))
    y_true, y_pred = evaluate_model(model, test_loader, device)

    macro_f1 = f1_score(y_true, y_pred, average='macro') * 100
    bal_acc = balanced_accuracy_score(y_true, y_pred) * 100
    cm = confusion_matrix(y_true, y_pred, normalize='true') * 100  # expressed as 0-100% percentage

    results[name] = {
        'Macro_F1': macro_f1,
        'Balanced_Acc': bal_acc,
        'cm': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }
    print(f"  -> {name}: Macro-F1 = {macro_f1:.2f}% | Balanced Accuracy = {bal_acc:.2f}%")

### 2.4 Generazione delle Matrici di Confusione Normalizzate (Task T2.6)
Tracciamo le 3 matrici di confusione normalizzate in percentuale (0–100%) con un'unica scala colore condivisa per il paper.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=True)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])

for idx, (name, data) in enumerate(results.items()):
    ax = axes[idx]
    cm = data['cm']
    sns.heatmap(
        cm,
        ax=ax,
        cmap='Blues',
        vmin=0,
        vmax=100,
        annot=False,
        cbar=(idx == 2),
        cbar_ax=cbar_ax if (idx == 2) else None,
        square=True
    )
    ax.set_title(f"{name}\n(Macro-F1: {data['Macro_F1']:.1f}% | BA: {data['Balanced_Acc']:.1f}%)", fontsize=12, fontweight='bold')
    ax.set_xlabel(r'Classe Predetta, $\hat{y} \in [0..12]$', fontsize=10)
    if idx == 0:
        ax.set_ylabel(r'Classe Reale di Rete, $y \in [0..12]$', fontsize=10)
cbar_ax.set_ylabel('Tasso di Corretta Classificazione [%]', fontsize=10)

plt.suptitle('Confronto Matrici di Confusione Normalizzate (%) - CIC-IoMT-2024 (13 Classi, T=10, Seed 0)', fontsize=14, y=1.02)
out_cm_png = FIGURES_DIR / 'confusion_matrices_benchmark.png'
out_cm_pdf = FIGURES_DIR / 'confusion_matrices_benchmark.pdf'
plt.savefig(out_cm_png, dpi=300, bbox_inches='tight')
plt.savefig(out_cm_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] Figure salvate in:\n  - {out_cm_png}\n  - {out_cm_pdf}")

### 2.5 Integrazione del Profiling di Latenza CPU Edge & Tabella Finale (Task T2.5)
Combiniamo i risultati di accuratezza con i tempi di risposta su CPU single-thread.

In [ ]:
import json
import re

latency_csv = METRICS_DIR / 'edge_latency_profiling.csv'
df_lat = pd.read_csv(latency_csv).set_index('Network')  # generated by scripts/profile_latency.py --all
if df_lat['Checkpoint'].astype(str).str.startswith('None').any():
    raise RuntimeError(f"{latency_csv} non e' misurato sui checkpoint addestrati: eseguire "
                       "'python scripts/profile_latency.py --all' dopo il Batch Job (Guida 14.3 regola 5).")

def train_summary(exp_id):
    """Epoca best model e Val Loss Min dal raw_log di main.py; tempo di training dal log di run_training.sh."""
    exp_dir = RESULTS_DIR / f"cic-iomt_scratch_{exp_id}"
    raw_logs = sorted(exp_dir.glob('raw_log-*.txt'))
    if not raw_logs:
        raise FileNotFoundError(f"Nessun raw_log-*.txt in {exp_dir}")
    with open(raw_logs[-1]) as f:
        valid = [(r['iter'], r['value']) for r in map(json.loads, f)
                 if r['name'] == 'loss' and r['group'] == 'valid']
    best_epoch, val_loss_min = min(valid, key=lambda p: p[1])
    m = re.search(r'TRAINING_TIME_S=(\d+)', (RESULTS_DIR / f"train_{exp_id}.log").read_text())
    if m is None:
        raise RuntimeError(f"TRAINING_TIME_S assente in train_{exp_id}.log: rilanciare scripts/run_training.sh")
    return best_epoch, val_loss_min, int(m.group(1))

benchmark_records = []
for name, data in results.items():
    lat = df_lat.loc[name]
    best_epoch, val_loss_min, train_s = train_summary(RUN_IDS[name])
    rec = {
        'Run_ID': RUN_IDS[name][:6],
        'Model': name,
        'Macro_F1_%': round(data['Macro_F1'], 2),
        'Balanced_Acc_%': round(data['Balanced_Acc'], 2),
        'Best_Epoch': best_epoch,
        'Val_Loss_Min': round(val_loss_min, 4),
        'Training_Time_min': round(train_s / 60, 1),
        'Params': int(lat['Params']),
        'Size_MB': float(lat['Size_MB']),
        'Latency_CPU_ms': float(lat['Latency_Median_ms']),
        'Latency_P95_ms': float(lat['Latency_P95_ms']),
        'Throughput_fps': float(lat['Throughput_fps'])
    }
    benchmark_records.append(rec)

df_benchmark = pd.DataFrame(benchmark_records)
df_benchmark.to_csv(METRICS_DIR / 'benchmark_metrics.csv', index=False)
print("\n======================================================================")
print(" TABELLA BENCHMARK COMPARATIVO FINALE (Fase 2 / BLK-1)")
print("======================================================================")
display(df_benchmark) if 'display' in globals() else print(df_benchmark)

### 2.6 Formattazione Risultati per `results/final_experimental_report.md`
Genera la riga Markdown pronta per essere riportata nel registro sperimentale (Sezione 11 di `results/final_experimental_report.md`).

In [ ]:
print("### Tabella Markdown per results/final_experimental_report.md (Sezione 11, Tabella 1):")
print("| Run ID | Architettura | Epoche | Macro-F1 (%) | Balanced Acc (%) | Epoca Best Model | Val Loss Min | Tempo Training |")
print("| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |")
for r in benchmark_records:
    print(f"| **{r['Run_ID']}** | `{r['Model']}` | 50 | **{r['Macro_F1_%']}%** | **{r['Balanced_Acc_%']}%** | {r['Best_Epoch']} | {r['Val_Loss_Min']} | {r['Training_Time_min']} min |")